In [0]:
# Visualização da Evolução no Tempo
display(df_tempo)

data,faturamento_total
2026-04-01,3750
2026-04-02,2850
2026-04-03,4000
2026-04-04,1800


In [0]:
# Visualização do faturamento por Categoria
display(df_categoria)

categoria,faturamento_total
Eletronico,10600
Acessorio,1800


In [0]:
# Visualização do faturamento por produto
display(spark.table("gold_vendas"))

produto,faturamento_total
Mouse,750
Monitor,3600
Teclado,1050
Notebook,7000


## Conclusão

Este projeto demonstra a construção de um pipeline de dados utilizando a arquitetura Medallion (Bronze, Silver e Gold).

Foram aplicadas técnicas de ingestão, transformação e agregação de dados utilizando PySpark e Delta Lake no Databricks.

Os dados foram estruturados para análise de faturamento por produto, categoria e evolução temporal.

In [0]:
# Evolução do Faturamento ao Longo do Tempo
df_tempo = df_silver.groupBy("data").agg(
    sum("total").alias("faturamento_total")
)

display(df_tempo)

data,faturamento_total
2026-04-01,3750
2026-04-02,2850
2026-04-03,4000
2026-04-04,1800


In [0]:
# Análise de Faturamento por Categoria
from pyspark.sql.functions import sum

df_silver = spark.table("silver_vendas")

df_categoria = df_silver.groupBy("categoria").agg(
    sum("total").alias("faturamento_total")
)

display(df_categoria)

categoria,faturamento_total
Eletronico,10600
Acessorio,1800


In [0]:
# Análise de Faturamento por Produto
df_gold = spark.table("gold_vendas")
display(df_gold)

produto,faturamento_total
Mouse,750
Monitor,3600
Teclado,1050
Notebook,7000


In [0]:
# Camada Gold. Criação de métricas agregadas para análise de negócio.
from pyspark.sql.functions import sum

df_silver = spark.table("silver_vendas")

df_gold = df_silver.groupBy("produto").agg(
    sum("total").alias("faturamento_total")
)

df_gold.write.format("delta").mode("overwrite").saveAsTable("gold_vendas")

In [0]:
# Camada Silver. Tratamento dos dados e criação da coluna de valor total.
df_bronze = spark.table("bronze_vendas")

from pyspark.sql.functions import col

df_silver = df_bronze.withColumn("total", col("quantidade") * col("preco"))

df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_vendas")

In [0]:
# Camada Bronze. Armazenamento dos dados brutos no formato Delta Lake.
df_spark.write.format("delta").mode("overwrite").saveAsTable("bronze_vendas")

In [0]:
## Ingestão de Dados (Criação do Dataset). Criação de um conjunto de dados de vendas para simulação do pipeline.
import pandas as pd

data = {
    "id_venda": [1,2,3,4,5,6,7,8],
    "data": ["2026-04-01","2026-04-01","2026-04-02","2026-04-02","2026-04-03","2026-04-03","2026-04-04","2026-04-04"],
    "produto": ["Notebook","Mouse","Teclado","Monitor","Mouse","Notebook","Monitor","Teclado"],
    "categoria": ["Eletronico","Acessorio","Acessorio","Eletronico","Acessorio","Eletronico","Eletronico","Acessorio"],
    "quantidade": [1,5,3,2,10,1,1,4],
    "preco": [3500,50,150,1200,50,3500,1200,150]
}

df = pd.DataFrame(data)
df

,id_venda,data,produto,categoria,quantidade,preco
0,1,2026-04-01,Notebook,Eletronico,1,3500
1,2,2026-04-01,Mouse,Acessorio,5,50
2,3,2026-04-02,Teclado,Acessorio,3,150
3,4,2026-04-02,Monitor,Eletronico,2,1200
4,5,2026-04-03,Mouse,Acessorio,10,50
5,6,2026-04-03,Notebook,Eletronico,1,3500
6,7,2026-04-04,Monitor,Eletronico,1,1200
7,8,2026-04-04,Teclado,Acessorio,4,150


# Pipeline de Engenharia de Dados - Arquitetura Medallion (Bronze, Silver, Gold)

## Tecnologias utilizadas
- Databricks
- PySpark
- SQL
- Delta Lake